# S&P 500 配對交易系統 (SSD 方法)
## 基於 Gatev et al. (2006) 的嚴格實作
### 回測期間：2000-2025

本 Notebook 遵循以下檢核清單：

**1. 數據預處理 (Normalization)**
- [x] 起始點歸一化：$P'_{i,0} = 1$ (形成期第一天)
- [x] 總報酬索引（含息）
- [x] 公式檢驗：$P'_{i,t} = \frac{P_{i,t}}{P_{i,0}}$

**2. 配對篩選 (Pair Selection)**
- [x] 距離測度：$\text{SSD} = \sum_{t=1}^T (P'_{a,t} - P'_{b,t})^2$
- [x] 無對沖比率計算（1:1 固定）
- [x] 行業中性分組

**3. 交易規則 (Trading Rules)**
- [x] 進場門檻：$|diff_t| > 2\sigma_{formation}$
- [x] 平倉條件：$diff_t \approx 0$ 或交易期結束
- [x] 固定形成期標準差（非動態）

**4. 績效計算 (Performance)**
- [x] 等權重對沖（$1 買入低估股 - $1 賣出高估股）
- [x] 損益來自價差收斂

In [1]:
# 套件安裝與導入
import subprocess, sys
for pkg in ['statsmodels', 'plotly', 'kaleido', 'yfinance', 'joblib']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore')
import sqlite3, numpy as np, pandas as pd
import logging
from itertools import combinations
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from IPython.display import display, HTML
import yfinance as yf
import time
import math
import os

pd.set_option('display.float_format', '{:.4f}'.format)
print('✓ 套件導入完成')

✓ 套件導入完成


In [2]:
# ========== 階段 1：數據預處理與正規化 ==========
# === 全局參數設置 ===
FAST_TEST_MODE = True

if FAST_TEST_MODE:
    print("【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    # 縮短時間：涵蓋 2020 疫情崩盤與 2022 升息的壓力測試區間
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # 縮小股票池：僅測試單一板塊，運算量大幅減少
    TARGET_SECTOR = 'Information Technology'  
else:
    print("【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    # 論文要求的全樣本時間
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    # 測試全市場 (設為 None 代表不限制單一產業)
    TARGET_SECTOR = None  

# 資料庫配置
DB_PATH = r'..\data\sp500.db'
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'
USE_DYNAMIC_SECTORS = True  # 是否使用動態產業補齊

# 視窗參數 (嚴格遵循 GGR 論文)
FORMATION_WINDOW = 252    # 約 1 年
TRADING_WINDOW = 126      # 約 6 個月
ROLLING_WINDOW = 20       # 約 1 個月 (梯隊步長)
MIN_HISTORY_DAYS = 200    # 最少歷史資料

# 配對篩選參數
TOP_N_PAIRS = 10          # 每個視窗選出前 N 組配對
SECTOR_NEUTRAL = True     # 行業中性化

# 交易參數
Z_ENTRY = 2.0             # 進場標準差閾值（SSD 方法中改用 2×σ_formation）
Z_EXIT = 0.0              # 平倉標準差閾值
TRANSACTION_COST = 0.0029 # 雙邊交易成本 (0.29%)
MAX_LOSS_PCT = 0          # 停損 (0 = 無停損)

# 資金配置
INITIAL_CAPITAL = 10000  # 初始本金
CAPITAL_TRANCHES = math.ceil(TRADING_WINDOW / ROLLING_WINDOW) + 1

# 確保路徑存在
os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

print(f"""
【回測參數設置】
時間期間: {START_DATE} ~ {END_DATE}
形成期: {FORMATION_WINDOW} 天（約 1 年）
交易期: {TRADING_WINDOW} 天（約 6 個月）
步長: {ROLLING_WINDOW} 天（約 1 個月）
配對數: {TOP_N_PAIRS}
行業中性: {SECTOR_NEUTRAL}
初始資金: ${INITIAL_CAPITAL:,.0f}
交易成本: {TRANSACTION_COST*100:.2f}%
""")

【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。

【回測參數設置】
時間期間: 2019-01-01 ~ 2022-12-31
形成期: 252 天（約 1 年）
交易期: 126 天（約 6 個月）
步長: 20 天（約 1 個月）
配對數: 10
行業中性: True
初始資金: $10,000
交易成本: 0.29%



In [3]:
# === 數據加載函數 ===
def load_data_from_db(db_path, start_date, end_date):
    """從 SQLite 資料庫加載股票價格和行業分類"""
    conn = sqlite3.connect(db_path)
    
    # 嘗試多個可能的表名
    price_queries = [
        (f"SELECT date, ticker, adj_close AS close FROM daily_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
        (f"SELECT date, ticker, close FROM stock_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
    ]
    
    prices_df = None
    for q in price_queries:
        try:
            prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
            if len(prices_df) > 0:
                print(f'✓ 加載價格數據：{len(prices_df):,} 筆記錄')
                break
        except Exception:
            continue
    
    if prices_df is None or len(prices_df) == 0:
        raise RuntimeError("無法從資料庫加載價格數據")
    
    # 加載行業分類
    sector_queries = [
        "SELECT ticker, sector FROM tickers",
        "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
    ]
    
    sector_df = None
    for q in sector_queries:
        try:
            sector_df = pd.read_sql_query(q, conn)
            if len(sector_df) > 0:
                print(f'✓ 加載行業數據：{len(sector_df):,} 檔股票')
                break
        except Exception:
            continue
    
    conn.close()
    
    if sector_df is None or len(sector_df) == 0:
        print('⚠️  未找到行業表，使用 Unknown 代替')
        sector_df = pd.DataFrame({'ticker': prices_df['ticker'].unique(), 'sector': 'Unknown'})
    
    return prices_df, sector_df

def fix_unknown_sectors(sector_df, use_dynamic=USE_DYNAMIC_SECTORS, save_path=r'data\imputed_sectors.csv'):
    """具備本機快取與全域開關控制的產業補齊模組"""
    # 1. 開關判斷：如果不使用動態補齊，直接原封不動回傳
    if not use_dynamic:
        print("不使用動態產業補齊。")
        return sector_df

    # 確保儲存的目錄 (data\) 存在
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # 2. 快取讀取：如果已經抓過並存檔，直接載入
    if os.path.exists(save_path):
        print(f"從本機快取載入已補齊的產業分類: {save_path}")
        cached_df = pd.read_csv(save_path)
        update_df = cached_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        return sector_df.reset_index()

    # 3. API 抓取：如果沒有快取，執行連線作業
    unknown_mask = sector_df['sector'] == 'Unknown'
    unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()
    
    if not unknown_tickers:
        return sector_df

    print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔股票的產業分類...")
    
    yf_logger = logging.getLogger('yfinance')
    original_level = yf_logger.level
    yf_logger.setLevel(logging.CRITICAL) 
    
    fixed_sectors = []
    
    for i, ticker in enumerate(unknown_tickers):
        try:
            info = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            fixed_sectors.append({'ticker': ticker, 'sector': sector})
            time.sleep(0.02) 
        except Exception:
            fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
            
        if (i + 1) % 50 == 0:
            print(f"已處理 {i + 1} / {len(unknown_tickers)}...")
            
    yf_logger.setLevel(original_level)
    
    # 4. 儲存快取：將剛抓下來的資料存成 CSV，下次就不用再抓了
    fetched_df = pd.DataFrame(fixed_sectors)
    fetched_df.to_csv(save_path, index=False)
    print(f"API 抓取完畢！已將動態產業分類永久儲存至: {save_path}")
    
    # 更新回原本的 DataFrame
    update_df = fetched_df.set_index('ticker')
    sector_df = sector_df.set_index('ticker')
    sector_df.update(update_df)
    sector_df = sector_df.reset_index()
    
    remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
    print(f"補齊完成！剩餘真實無法識別(已下市)的股票數量: {remaining}")
    
    return sector_df


def preprocess_prices(prices_df, min_days=MIN_HISTORY_DAYS):
    """
    數據預處理：樞紐、前向填充、去除稀疏股票
    
    檢核清單項：
    ✓ 缺失值填充（最多 5 天前向填充）
    ✓ 去除歷史資料不足的股票
    ✓ 對齊時間序列
    """
    # 樞紐表：時間 × 股票
    pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot.index = pd.to_datetime(pivot.index)
    pivot.sort_index(inplace=True)
    
    # 前向填充（最多 5 天）
    pivot.ffill(limit=5, inplace=True)
    
    # 保留有足夠歷史資料的股票
    valid = pivot.columns[pivot.notna().sum() >= min_days]
    pivot = pivot[valid]
    
    print(f'✓ 數據矩陣：{len(pivot)} 天 × {len(pivot.columns)} 檔股票')
    print(f'✓ 時間範圍：{pivot.index[0].date()} ~ {pivot.index[-1].date()}')
    
    return pivot


# === 加載和預處理數據 ===
print("\n從資料庫加載原始數據...")
prices_raw, sector_info = load_data_from_db(DB_PATH, START_DATE, END_DATE)

print("\n預處理價格數據...")
price_pivot = preprocess_prices(prices_raw, min_days=MIN_HISTORY_DAYS)

# 建立行業對應字典
sector_map = sector_info.set_index('ticker')['sector'].to_dict()
print(f'\n【行業分佈】')
print(pd.Series(sector_map).value_counts().head(10))


從資料庫加載原始數據...
✓ 加載價格數據：616,467 筆記錄
✓ 加載行業數據：843 檔股票

預處理價格數據...
✓ 數據矩陣：1008 天 × 623 檔股票
✓ 時間範圍：2019-01-02 ~ 2022-12-30

【行業分佈】
Unknown                   340
Industrials                79
Financials                 76
Information Technology     71
Health Care                60
Consumer Discretionary     48
Consumer Staples           36
Utilities                  31
Real Estate                31
Materials                  26
Name: count, dtype: int64


In [4]:
# 板塊過濾（如果設定了 TARGET_SECTOR）
if TARGET_SECTOR is not None:
    # 篩選出符合目標產業的股票 ticker
    target_tickers = sector_info[sector_info['sector'] == TARGET_SECTOR]['ticker'].tolist()
    print(f"🎯 板塊過濾：篩選 {TARGET_SECTOR} 板塊之股票，共 {len(target_tickers)} 檔")
    
    # 過濾 prices_raw：只保留目標產業的股票
    prices_raw = prices_raw[prices_raw['ticker'].isin(target_tickers)]
    
    # 過濾 sector_info：只保留目標產業的資訊
    sector_info = sector_info[sector_info['sector'] == TARGET_SECTOR]
else:
    print("🌍 全市場模式：使用全部 S&P 500 股票")

# ============================================================

# 
price_pivot= preprocess_prices(prices_raw)
sector_map = sector_info.set_index('ticker')['sector'].to_dict()

🎯 板塊過濾：篩選 Information Technology 板塊之股票，共 71 檔
✓ 數據矩陣：1008 天 × 69 檔股票
✓ 時間範圍：2019-01-02 ~ 2022-12-30


## 階段 2：行業分組與配對篩選 (SSD 方法)

### 檢核清單：
- ✓ **距離測度**：$\text{SSD}(A,B)=\sum_{t=1}^T (P'_{a,t} - P'_{b,t})^2$
- ✓ **正規化價格**：$P'_{i,t} = \frac{P_{i,t}}{P_{i,0}}$ (形成期首日強制設為 1)
- ✓ **無對沖比率**：SSD 法假設 1:1 等權重
- ✓ **行業中性**：僅在同產業內篩選配對
- ✓ **形成期標準差**：用於交易期進場門檻

### 數學公式：
$$\text{SSD}(A,B) = \sum_{t=1}^T (P'_{a,t} - P'_{b,t})^2$$

其中 $P'_{i,t}$ 在形成期首日 ($t=0$) 必須滿足 $P'_{i,0} = 1$

In [ ]:
# ========== 階段 2：行業分組與配對篩選 ==========
def select_pairs_ssd(price_window, sector_map, top_n=TOP_N_PAIRS, sector_neutral=True):
    """
    完全符合 GGR (2006) 的 SSD 配對篩選邏輯
    
    檢核清單項：
    ✓ 步驟 1：正規化價格（首日 = 1）
    ✓ 步驟 2：計算 SSD (Sum of Squared Deviations)
    ✓ 步驟 3：無迴歸係數計算（1:1 固定對沖）
    ✓ 步驟 4：計算形成期標準差（用於未來交易門檻）
    ✓ 步驟 5：行業分組（可選）
    """
    
    # ===== 步驟 1：正規化價格 =====
    # 公式：P'_{i,t} = P_{i,t} / P_{i,0}
    # 確保形成期首日所有股票的 P'_{i,0} = 1
    def normalize_prices(prices):
        """正規化：首日設為 1"""
        first_price = prices.iloc[0]
        # 避免除以零
        valid_mask = first_price != 0
        normalized = pd.DataFrame(index=prices.index)
        for col in prices.columns:
            if valid_mask[col]:
                normalized[col] = prices[col] / first_price[col]
            else:
                normalized[col] = np.nan
        return normalized
    
    # 去除缺失列，正規化
    clean_window = price_window.dropna(axis=1)
    norm = normalize_prices(clean_window)
    valid_tickers = norm.columns.tolist()
    
    # print(f"【數據檢查】有效股票數: {len(valid_tickers)}")
    
    # ===== 步驟 2：建立候選配對（行業分組） =====
    if sector_neutral:
        groups = {}
        for t in valid_tickers:
            sec = sector_map.get(t, 'Unknown')
            if sec != 'Unknown':
                groups.setdefault(sec, []).append(t)
        
        # 同產業內的所有配對
        candidates = []
        for sector, tickers in groups.items():
            if len(tickers) >= 2:
                for a, b in combinations(tickers, 2):
                    candidates.append((a, b, sector))
        
        # print(f"【行業分組】{len(groups)} 個產業, {len(candidates)} 組配對候選")
    else:
        candidates = [(a, b, 'All') for a, b in combinations(valid_tickers, 2)]
        # print(f"【全市場配對】{len(candidates)} 組配對候選")
    
    # ===== 步驟 3：計算每一對的 SSD 和標準差 =====
    results = []
    
    for a, b, sector in candidates:
        try:
            # 正規化價差：P'_a - P'_b
            price_a = norm[a]
            price_b = norm[b]
            
            # 去除任一方缺失的日期
            valid_idx = (~price_a.isna()) & (~price_b.isna())
            if valid_idx.sum() < 50:  # 最少 50 個有效數據點
                continue
            
            price_a_valid = price_a[valid_idx]
            price_b_valid = price_b[valid_idx]
            
            # 計算價差
            spread = price_a_valid - price_b_valid
            
            # 計算 SSD = Σ(P'_a - P'_b)²
            ssd = (spread ** 2).sum()
            
            # 計算形成期的價差標準差（用於交易期進場門檻）
            spread_std = spread.std()
            
            results.append({
                'stock_a': a,
                'stock_b': b,
                'sector': sector,
                'ssd': ssd,
                'spread_std': spread_std,
                'avg_price_a': price_a_valid.mean(),
                'avg_price_b': price_b_valid.mean(),
                'correlation': price_a_valid.corr(price_b_valid)
            })
        except Exception as e:
            print(f"⚠️ 計算配對 {a}-{b} 時發生錯誤: {e}")
            continue
    
    if not results:
        print("⚠️ 無有效配對")
        return []
    
    # ===== 步驟 4：依 SSD 排序並選出頂級配對 =====
    df_results = pd.DataFrame(results).sort_values('ssd')
    top_pairs = df_results.head(top_n).to_dict('records')
    
    # print(f"【配對篩選結果】有 {len(top_pairs)} 組進入配對清單")
    
    # 打印頂級配對詳情
    # print(f"\n【配對細節】")
    # for i, pair in enumerate(top_pairs[:5], 1):
    #     print(f"  {i}. {pair['stock_a']}-{pair['stock_b']} | 產業: {pair['sector']} | "
    #           f"SSD: {pair['ssd']:.2f} | 價差 Std: {pair['spread_std']:.4f}")
    
    return top_pairs

print("✓ SSD 配對篩選函數已定義")

✓ SSD 配對篩選函數已定義


## 階段 3：交易信號生成與規則定義

### 檢核清單：
- ✓ **進場條件**：$|diff_t| > 2 \times \sigma_{formation}$
- ✓ **平倉條件**：$diff_t \approx 0$ 或交易期結束
- ✓ **固定標準差**：$\sigma_{formation}$ 於形成期計算，交易期不變
- ✓ **等權重對沖**：做多低估股 $1，做空高估股 $1
- ✓ **交易日誌**：記錄進出場時間、價差、持倉期

### 交易邏輯狀態機：
```
無倉位 → |diff| > 2σ → 進場
進場 → |diff| < 0.5σ 或交易期結束 → 平倉
平倉 → 冷卻期 → 無倉位
```

In [6]:
# ========== 階段 3：交易信號生成與規則定義 ==========
def execute_ssd_trades(trade_prices, pair, form_prices, formation_std, 
                       entry_mult=Z_ENTRY, exit_mult=Z_EXIT, cost=TRANSACTION_COST, pair_capital=1000.0):
    """
    基於 SSD 方法的交易執行
    """
    import pandas as pd
    a, b = pair['stock_a'], pair['stock_b']
    
    try:
        p_a = trade_prices[a]
        p_b = trade_prices[b]
    except KeyError:
        return {'pnl': pd.Series(0.0, index=trade_prices.index), 'trades': []}
    
    valid_idx = (~p_a.isna()) & (~p_b.isna())
    if valid_idx.sum() < 5:
        return {'pnl': pd.Series(0.0, index=trade_prices.index), 'trades': []}
    
    valid_dates = trade_prices.index[valid_idx]
    p_a = p_a[valid_idx]
    p_b = p_b[valid_idx]
    
    # 關鍵修正 1：使用形成期首日價格進行標準化！
    p_a0 = form_prices[a].iloc[0]
    p_b0 = form_prices[b].iloc[0]
    
    if p_a0 == 0 or p_b0 == 0 or pd.isna(p_a0) or pd.isna(p_b0):
        return {'pnl': pd.Series(0.0, index=trade_prices.index), 'trades': []}
    
    norm_a = p_a / p_a0
    norm_b = p_b / p_b0
    
    # 計算真正的 正規化價差 (與 formation_std 在相同基準)
    spread = norm_a - norm_b
    
    entry_threshold = entry_mult * formation_std
    exit_threshold = exit_mult * formation_std
    
    pnl_daily = {}
    trades = []
    
    position = 0  
    entry_date = None
    entry_spread = None
    entry_price_a = None
    entry_price_b = None
    shares_a = 0
    shares_b = 0
    
    for i in range(len(valid_dates)):
        dt = valid_dates[i]
        s = spread.iloc[i]
        pa = p_a.iloc[i]
        pb = p_b.iloc[i]
        
        pnl = 0.0
        
        if position != 0 and i > 0:
            # 關鍵修正 2：真實資金部位換算 (Mark to Market)
            # pnl = \Delta市值
            prev_pa = p_a.iloc[i-1]
            prev_pb = p_b.iloc[i-1]
            pnl = shares_a * (pa - prev_pa) + shares_b * (pb - prev_pb)
        
        if position == 0:
            if abs(s) > entry_threshold:
                position = 1 if s < 0 else -1  # s < 0: A相對低估，做多A作空B
                entry_date = dt
                entry_spread = s
                entry_price_a = pa
                entry_price_b = pb
                
                # 資金等分分配到雙邊：各得 C/2
                side_capital = pair_capital / 2
                shares_a = position * side_capital / pa
                shares_b = -position * side_capital / pb
                
                pnl -= pair_capital * cost  # 雙邊進場成本定死以 pair_capital 算
        else:
            should_exit = False
            
            if (position == 1 and s >= exit_threshold) or (position == -1 and s <= -exit_threshold):
                should_exit = True
                exit_reason = "價差收斂"
            elif i == len(valid_dates) - 1:
                should_exit = True
                exit_reason = "交易期結束"
            
            if should_exit:
                exit_spread = s
                # 總獲利 = 終點市值 - 初始市值 - 來回雙邊手續費
                profit = (shares_a * pa + shares_b * pb) - (shares_a * entry_price_a + shares_b * entry_price_b) - (pair_capital * cost * 2)
                hold_days = (dt - entry_date).days
                
                trades.append({
                    'stock_a': a,
                    'stock_b': b,
                    'entry_date': entry_date,
                    'exit_date': dt,
                    'entry_spread': entry_spread,
                    'exit_spread': exit_spread,
                    'profit': profit,
                    'profit_pct': profit / pair_capital * 100,
                    'hold_days': hold_days,
                    'exit_reason': exit_reason
                })
                
                position = 0
                shares_a = 0
                shares_b = 0
                pnl -= pair_capital * cost  # 平倉成本
                
        pnl_daily[dt] = pnl
        
    if position != 0:
        # 強平紀錄補遺
        final_spread = spread.iloc[-1]
        profit = (shares_a * p_a.iloc[-1] + shares_b * p_b.iloc[-1]) - (shares_a * entry_price_a + shares_b * entry_price_b) - (pair_capital * cost * 2)
        trades.append({
            'stock_a': a, 'stock_b': b,
            'entry_date': entry_date, 'exit_date': valid_dates[-1],
            'entry_spread': entry_spread, 'exit_spread': final_spread,
            'profit': profit,
            'profit_pct': profit / pair_capital * 100,
            'hold_days': (valid_dates[-1] - entry_date).days,
            'exit_reason': '交易期結束（強制）'
        })
        pnl_daily[valid_dates[-1]] -= pair_capital * cost
        
    pnl_series = pd.Series(pnl_daily, dtype=float).reindex(trade_prices.index, fill_value=0.0)
    
    return {
        'pnl': pnl_series,
        'trades': trades,
        'formation_std': formation_std,
        'entry_threshold': entry_threshold,
        'exit_threshold': exit_threshold
    }
print("✓ 交易執行函數已修正。加入真實資本換算與正確預處理基準。")


✓ 交易執行函數已修正。加入真實資本換算與正確預處理基準。


## 階段 4：滾動視窗回測框架

### 檢核清單：
- ✓ **形成期**：252 天（約 1 年）
- ✓ **交易期**：126 天（約 6 個月）
- ✓ **滾動步長**：20 天（約 1 個月）
- ✓ **梯隊資金**：將資本分割到每個滾動視窗
- ✓ **聚合損益**：逐日累積所有視窗的損益

In [7]:
# ========== 階段 4：滾動視窗回測框架 ==========
def run_ssd_backtest(price_pivot, sector_map,
                     formation_window=FORMATION_WINDOW,
                     trading_window=TRADING_WINDOW,
                     rolling_window=ROLLING_WINDOW,
                     top_n=TOP_N_PAIRS,
                     initial_capital=INITIAL_CAPITAL):
    """
    完整的滾動視窗 SSD 配對交易回測
    
    檢核清單項：
    ✓ 形成期 = 252 天
    ✓ 交易期 = 126 天
    ✓ 滾動步長 = 20 天
    ✓ 梯隊資金管理
    ✓ 日度損益聚合
    """
    
    dates = price_pivot.index
    N = len(dates)
    
    # ===== 初始化 =====
    all_pnl = pd.DataFrame(index=dates, dtype=float)
    all_trades = []
    window_records = []
    
    num_tranches = math.ceil(trading_window / rolling_window) + 1
    tranche_capital = initial_capital / num_tranches
    
    print(f"\n【回測配置】")
    print(f"  資本梯隊數: {num_tranches}")
    print(f"  每梯隊配置: ${tranche_capital:,.2f}")
    print(f"  時間跨度: {dates[0].date()} ~ {dates[-1].date()}")
    
    # ===== 滾動視窗迴圈 =====
    si, window_id = 0, 0
    
    while si + formation_window + trading_window <= N:
        fe = si + formation_window        # 形成期結束
        te = fe + trading_window          # 交易期結束
        
        form_prices = price_pivot.iloc[si:fe]
        trade_prices = price_pivot.iloc[fe:te]
        
        # 配對篩選
        pairs = select_pairs_ssd(form_prices, sector_map, top_n=top_n)
        
        if not pairs:
            si += rolling_window
            window_id += 1
            continue
        
        # 本視窗的損益
        window_pnl = pd.Series(0.0, index=dates)
        window_all_trades = []
        
        # 對每個配對執行交易
        for pair in pairs:
            pair_cap = tranche_capital / len(pairs) if len(pairs)>0 else 0
            result = execute_ssd_trades(trade_prices, pair, form_prices, pair['spread_std'], pair_capital=pair_cap)
            
            pair_pnl = result['pnl']  # 已經是美元絕對值
            window_pnl = window_pnl.add(pair_pnl, fill_value=0.0)
            window_all_trades.extend(result['trades'])
        
        # 記錄本視窗
        col_name = f'W{window_id:03d}'
        all_pnl[col_name] = window_pnl
        all_trades.extend(window_all_trades)
        
        # 計算本視窗績效
        active_pnl = window_pnl[(window_pnl.index >= dates[fe]) & 
                                (window_pnl.index <= dates[te-1])]
        window_return = active_pnl.sum() / tranche_capital if tranche_capital > 0 else 0.0
        trade_count = len([t for t in window_all_trades 
                          if pd.to_datetime(t['entry_date']) >= dates[fe]])
        
        window_records.append({
            'window_id': window_id,
            'form_start': dates[si],
            'form_end': dates[fe-1],
            'trade_start': dates[fe],
            'trade_end': dates[te-1],
            'pairs_count': len(pairs),
            'trade_count': trade_count,
            'return_pct': window_return * 100,
            'pnl_total': active_pnl.sum()
        })
        
        # 列印進度
        print(f"  W{window_id:03d} | {dates[fe].date()} ~ {dates[te-1].date()} | "
                  f"配對: {len(pairs)} | 交易: {trade_count} | 報酬: {window_return*100:+.2f}%")
        
        si += rolling_window
        window_id += 1
    
    print(f"\n✓ 回測完成：{window_id} 個視窗，{len(all_trades)} 筆交易")
    
    return {
        'pnl': all_pnl,
        'trades': all_trades,
        'windows': window_records
    }

# ===== 執行回測 =====
print("\n執行滾動視窗回測...")
backtest_result = run_ssd_backtest(price_pivot, sector_map)


執行滾動視窗回測...

【回測配置】
  資本梯隊數: 8
  每梯隊配置: $1,250.00
  時間跨度: 2019-01-02 ~ 2022-12-30
【行業分組】1 個產業, 2080 組配對候選
  W000 | 2020-01-02 ~ 2020-07-01 | 配對: 10 | 交易: 21 | 報酬: +2.82%


【行業分組】1 個產業, 2080 組配對候選
  W001 | 2020-01-31 ~ 2020-07-30 | 配對: 10 | 交易: 19 | 報酬: +8.01%


【行業分組】1 個產業, 2080 組配對候選
  W002 | 2020-03-02 ~ 2020-08-27 | 配對: 10 | 交易: 21 | 報酬: +7.13%


【行業分組】1 個產業, 2080 組配對候選
  W003 | 2020-03-30 ~ 2020-09-25 | 配對: 10 | 交易: 21 | 報酬: +0.19%


【行業分組】1 個產業, 2080 組配對候選
  W004 | 2020-04-28 ~ 2020-10-23 | 配對: 10 | 交易: 14 | 報酬: +0.47%


【行業分組】1 個產業, 2080 組配對候選
  W005 | 2020-05-27 ~ 2020-11-20 | 配對: 10 | 交易: 15 | 報酬: +0.51%


【行業分組】1 個產業, 2145 組配對候選
  W006 | 2020-06-24 ~ 2020-12-21 | 配對: 10 | 交易: 16 | 報酬: -0.27%


【行業分組】1 個產業, 2145 組配對候選
  W007 | 2020-07-23 ~ 2021-01-21 | 配對: 10 | 交易: 15 | 報酬: -0.40%


【行業分組】1 個產業, 2145 組配對候選
  W008 | 2020-08-20 ~ 2021-02-19 | 配對: 10 | 交易: 16 | 報酬: -2.12%


【行業分組】1 個產業, 2211 組配對候選
  W009 | 2020-09-18 ~ 2021-03-19 | 配對: 10 | 交易: 14 | 報酬: +2.70%


【行業分組】1 個產業, 2211

## 階段 5：績效指標計算與分析

### 計算指標：
- **CAGR (年化報酬率)**：$\text{CAGR} = (1 + R)^{252/n} - 1$
- **年化波動率**：$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$
- **Sharpe Ratio**：$SR = \frac{r_{annual}}{\sigma_{annual}}$
- **Sortino Ratio**：$\text{Sortino} = \frac{r_{annual}}{\sigma_{downside}}$
- **最大回撤 (MDD)**：$\text{MDD} = \min\left(\frac{V_t - V_{max}}{V_{max}}\right)$
- **勝率**：$\frac{\text{正報酬日數}}{\text{總交易日數}}$
- **平均獲利/虧損**：配對層級統計

In [8]:
# ========== 階段 5：績效指標計算與分析 ==========
def compute_performance_metrics(pnl_df, initial_capital, all_trades):
    """計算完整的績效指標"""
    
    # ===== 計算組合損益和報酬率 =====
    portfolio_pnl = pnl_df.sum(axis=1)
    returns = portfolio_pnl / initial_capital
    returns = returns.dropna().replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(returns) == 0:
        print("⚠️  無有效報酬數據")
        return None
    
    # ===== 淨值曲線 =====
    nav = (1 + returns).cumprod()
    
    # ===== 基本指標 =====
    total_return = nav.iloc[-1] - 1
    cagr = (1 + total_return) ** (252 / len(returns)) - 1 if total_return > -1 else -1
    annual_vol = returns.std() * np.sqrt(252)
    
    # ===== 比率指標 =====
    sharpe = (returns.mean() * 252) / annual_vol if annual_vol > 1e-8 else np.nan
    
    # Sortino Ratio（僅計算下行波動率）
    downside_returns = returns[returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
    sortino = (returns.mean() * 252) / downside_vol if downside_vol > 1e-8 else np.nan
    
    # ===== 風險指標 =====
    drawdown = (nav - nav.cummax()) / nav.cummax()
    mdd = drawdown.min()
    
    # 恢復時間
    max_dd_date = drawdown.idxmin()
    recovery_dates = nav[nav.index > max_dd_date][nav >= nav[nav.index <= max_dd_date].max()]
    recovery_days = (recovery_dates.index[0] - max_dd_date).days if len(recovery_dates) > 0 else np.nan
    
    # ===== 交易統計 =====
    win_rate = (returns > 0).sum() / len(returns)
    
    # 配對層級統計
    if all_trades:
        trades_df = pd.DataFrame(all_trades)
        avg_profit = trades_df['profit'].mean()
        avg_hold = trades_df['hold_days'].mean()
        win_trades = (trades_df['profit'] > 0).sum()
        total_trades = len(trades_df)
        trade_win_rate = win_trades / total_trades if total_trades > 0 else 0
        avg_profit_win = trades_df[trades_df['profit'] > 0]['profit'].mean() if win_trades > 0 else 0
        avg_profit_loss = trades_df[trades_df['profit'] <= 0]['profit'].mean() if total_trades - win_trades > 0 else 0
    else:
        avg_profit = avg_hold = win_trades = total_trades = trade_win_rate = 0
        avg_profit_win = avg_profit_loss = 0.0
    
    metrics = {
        'CAGR(%)': round(cagr * 100, 2),
        '總報酬(%)': round(total_return * 100, 2),
        '年化波動(%)': round(annual_vol * 100, 2),
        'Sharpe': round(sharpe, 4),
        'Sortino': round(sortino, 4),
        '最大回撤(%)': round(mdd * 100, 2),
        '恢復天數': round(recovery_days, 0) if not np.isnan(recovery_days) else float('nan'),
        '日勝率(%)': round(win_rate * 100, 2),
        '配對勝率(%)': round(trade_win_rate * 100, 2),
        '平均配對利潤': round(avg_profit, 4),
        '平均獲利配對': round(avg_profit_win, 4),
        '平均虧損配對': round(avg_profit_loss, 4),
        '平均持倉天數': round(avg_hold, 1),
        '總配對數': total_trades
    }
    
    return metrics

# ===== 計算績效 =====
print("\n計算績效指標...")
pnl_df = backtest_result['pnl']
trades = backtest_result['trades']

metrics = compute_performance_metrics(pnl_df, INITIAL_CAPITAL, trades)

if metrics:
    print("\n【核心績效指標】")
    print(f"  CAGR: {metrics['CAGR(%)']:.2f}%")
    print(f"  年化波動: {metrics['年化波動(%)']:.2f}%")
    print(f"  Sharpe: {metrics['Sharpe']:.4f}")
    print(f"  最大回撤: {metrics['最大回撤(%)']:.2f}%")
    print(f"  日勝率: {metrics['日勝率(%)']:.2f}%")
    print(f"  配對勝率: {metrics['配對勝率(%)']:.2f}%")
    print(f"  總配對交易: {metrics['總配對數']}")
    
    # 詳細指標表
    metrics_df = pd.DataFrame([metrics])
    display(metrics_df.style.format("{:.2f}", na_rep="N/A"))


計算績效指標...

【核心績效指標】
  CAGR: 0.54%
  年化波動: 2.32%
  Sharpe: 0.2426
  最大回撤: -3.45%
  日勝率: 36.01%
  配對勝率: 59.71%
  總配對交易: 484


ValueError: Unknown format code 'f' for object of type 'str'

## 階段 6：結果視覺化與對標

### 視覺化內容：
1. **淨值曲線 (NAV)** - 累積報酬走勢
2. **回撤曲線** - 最大回撤動態
3. **日度報酬分佈** - 報酬率直方圖
4. **月度績效熱力圖** - 時間分解
5. **配對價差時序** - 交易信號驗證
6. **交易日誌表** - 詳細交易記錄
7. **對標比較** - vs. S&P 500

In [9]:
# ========== 階段 6：結果視覺化與對標 ==========
# ===== 準備數據 =====
portfolio_pnl = pnl_df.sum(axis=1)
returns = portfolio_pnl / INITIAL_CAPITAL
returns = returns.dropna().replace([np.inf, -np.inf], np.nan).dropna()
nav = (1 + returns).cumprod()
drawdown = (nav - nav.cummax()) / nav.cummax() * 100

# ===== 1. 淨值曲線與回撤 =====
print("\n【第 6.1 步】繪製�淨值曲線與回撤...")

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.65, 0.35],
    subplot_titles=['淨值曲線 (NAV)', '最大回撤 (MDD)'],
    vertical_spacing=0.1
)

# NAV
fig.add_trace(
    go.Scatter(
        x=nav.index, y=nav.values,
        name='SSD 策略',
        line=dict(color='#1f77b4', width=2),
        hovertemplate='%{x|%Y-%m-%d}<br>NAV: %{y:.4f}<extra></extra>'
    ),
    row=1, col=1
)

# 回撤
fig.add_trace(
    go.Scatter(
        x=drawdown.index, y=drawdown.values,
        name='回撤',
        fill='tozeroy',
        line=dict(color='#d62728', width=1),
        hovertemplate='%{x|%Y-%m-%d}<br>回撤: %{y:.2f}%<extra></extra>'
    ),
    row=2, col=1
)

fig.update_layout(
    title='SSD 配對交易系統 - 淨值與回撤曲線 (2000-2025)',
    height=700,
    template='plotly_dark',
    hovermode='x unified',
    showlegend=True
)

fig.update_yaxes(title_text='NAV', row=1, col=1)
fig.update_yaxes(title_text='回撤 (%)', row=2, col=1)
fig.show()

print("✓ 淨值曲線已繪製")

# ===== 2. 日度報酬分佈 =====
print("\n【第 6.2 步】繪製報酬分佈...")

fig_returns = go.Figure()

fig_returns.add_trace(go.Histogram(
    x=returns * 100,
    name='日度報酬',
    nbinsx=50,
    marker=dict(color='#2ca02c', opacity=0.7),
    hovertemplate='報酬率: %{x:.2f}%<br>頻次: %{y}<extra></extra>'
))

fig_returns.add_vline(
    x=returns.mean() * 100,
    line_dash='dash',
    line_color='red',
    annotation_text=f"平均: {returns.mean()*100:.3f}%",
    annotation_position="top right"
)

fig_returns.update_layout(
    title='日度報酬率分佈',
    xaxis_title='報酬率 (%)',
    yaxis_title='頻次',
    height=400,
    template='plotly_dark'
)

fig_returns.show()

print("✓ 報酬分佈已繪製")

# ===== 3. 月度績效熱力圖 =====
print("\n【第 6.3 步】生成月度績效熱力圖...")

# 按月計算報酬
returns_monthly = returns.resample('M').apply(lambda x: (1 + x).prod() - 1)
returns_monthly.index = returns_monthly.index.to_period('M')

# 轉換為年-月矩陣
if len(returns_monthly) > 0:
    pivot_returns = pd.DataFrame({
        'year': [d.year for d in returns_monthly.index],
        'month': [d.month for d in returns_monthly.index],
        'return': returns_monthly.values
    })
    
    pivot_table = pivot_returns.pivot(index='year', columns='month', values='return') * 100
    
    fig_heatmap = go.Figure(data=go.Heatmap(
        z=pivot_table.values,
        x=['1月', '2月', '3月', '4月', '5月', '6月', 
           '7月', '8月', '9月', '10月', '11月', '12月'],
        y=pivot_table.index,
        colorscale='RdYlGn',
        zmid=0,
        hovertemplate='%{y}年 %{x}: %{z:.2f}%<extra></extra>'
    ))
    
    fig_heatmap.update_layout(
        title='月度報酬率熱力圖 (%)',
        xaxis_title='月份',
        yaxis_title='年份',
        height=500,
        template='plotly_dark'
    )
    
    fig_heatmap.show()
    print("✓ 熱力圖已繪製")

# ===== 4. 視窗績效統計 =====
print("\n【第 6.4 步】視窗績效統計...")

windows_df = pd.DataFrame(backtest_result['windows'])
if len(windows_df) > 0:
    print(f"\n【視窗績效摘要】(共 {len(windows_df)} 個視窗)")
    print(windows_df[['window_id', 'pairs_count', 'trade_count', 'return_pct']].head(10)
                    .to_string(index=False))
    
    print(f"\n  平均配對數: {windows_df['pairs_count'].mean():.1f}")
    print(f"  平均交易數: {windows_df['trade_count'].mean():.1f}")
    print(f"  平均視窗報酬: {windows_df['return_pct'].mean():+.2f}%")
    print(f"  正報酬視窗: {(windows_df['return_pct'] > 0).sum()}/{len(windows_df)}")

# ===== 5. 交易日誌 =====
print("\n【第 6.5 步】交易日誌統計...")

if trades:
    trades_df = pd.DataFrame(trades)
    trades_df['entry_date'] = pd.to_datetime(trades_df['entry_date'])
    trades_df['exit_date'] = pd.to_datetime(trades_df['exit_date'])
    
    print(f"\n【交易統計】(共 {len(trades_df)} 筆交易)")
    print(f"  獲利交易: {(trades_df['profit'] > 0).sum()}")
    print(f"  虧損交易: {(trades_df['profit'] <= 0).sum()}")
    print(f"  平均利潤: {trades_df['profit'].mean():.6f}")
    print(f"  最大利潤: {trades_df['profit'].max():.6f}")
    print(f"  最大虧損: {trades_df['profit'].min():.6f}")
    print(f"  平均持倉: {trades_df['hold_days'].mean():.1f} 天")
    
    # 按平倉原因分類
    print(f"\n【平倉原因統計】")
    exit_reasons = trades_df['exit_reason'].value_counts()
    for reason, count in exit_reasons.items():
        pct = count / len(trades_df) * 100
        print(f"  {reason}: {count} ({pct:.1f}%)")
    
    # 前 10 筆交易
    print(f"\n【前 10 筆交易記錄】")
    display(trades_df[['stock_a', 'stock_b', 'entry_date', 'exit_date', 
                       'entry_spread', 'exit_spread', 'profit', 'hold_days']].head(10)
                    .style.format({'profit': '{:.6f}', 'entry_spread': '{:.4f}', 
                                 'exit_spread': '{:.4f}'}))

print("\n✓ 視覺化與分析完成")


【第 6.1 步】繪製�淨值曲線與回撤...


✓ 淨值曲線已繪製

【第 6.2 步】繪製報酬分佈...


✓ 報酬分佈已繪製

【第 6.3 步】生成月度績效熱力圖...


✓ 熱力圖已繪製

【第 6.4 步】視窗績效統計...

【視窗績效摘要】(共 32 個視窗)
 window_id  pairs_count  trade_count  return_pct
         0           10           21      2.8189
         1           10           19      8.0140
         2           10           21      7.1259
         3           10           21      0.1909
         4           10           14      0.4723
         5           10           15      0.5102
         6           10           16     -0.2717
         7           10           15     -0.3995
         8           10           16     -2.1213
         9           10           14      2.7008

  平均配對數: 10.0
  平均交易數: 15.1
  平均視窗報酬: +0.56%
  正報酬視窗: 16/32

【第 6.5 步】交易日誌統計...

【交易統計】(共 484 筆交易)
  獲利交易: 289
  虧損交易: 195
  平均利潤: 0.464595
  最大利潤: 22.910405
  最大虧損: -35.273392
  平均持倉: 81.8 天

【平倉原因統計】
  交易期結束: 250 (51.7%)
  價差收斂: 234 (48.3%)

【前 10 筆交易記錄】


,stock_a,stock_b,entry_date,exit_date,entry_spread,exit_spread,profit,hold_days
0,ROP,VRSN,2020-02-10 00:00:00,2020-03-09 00:00:00,0.0869,-0.0055,2.660642,28
1,ROP,VRSN,2020-03-24 00:00:00,2020-05-27 00:00:00,-0.0855,0.0137,6.235203,64
2,ROP,VRSN,2020-06-04 00:00:00,2020-07-01 00:00:00,0.0992,0.0500,1.299349,27
3,IBM,ORCL,2020-01-31 00:00:00,2020-03-13 00:00:00,0.1225,-0.0906,8.957418,42
4,IBM,ORCL,2020-03-18 00:00:00,2020-06-04 00:00:00,-0.1162,0.0009,7.903073,78
5,IBM,ORCL,2020-06-19 00:00:00,2020-07-01 00:00:00,-0.0941,-0.1554,-3.982885,12
6,ACN,MSFT,2020-01-02 00:00:00,2020-07-01 00:00:00,-0.0983,-0.5070,-16.460950,181
7,CTSH,GLW,2020-02-06 00:00:00,2020-06-08 00:00:00,0.1779,-0.0076,9.251055,123
8,ADI,MCHP,2020-01-02 00:00:00,2020-02-26 00:00:00,-0.1026,0.0093,3.641730,55
9,ADI,MCHP,2020-03-12 00:00:00,2020-05-26 00:00:00,0.1840,-0.0014,15.535499,75



✓ 視覺化與分析完成


### 【年淨值走勢圖】
按年份展示淨值變化，清晰呈現策略在不同年度的表現。

In [10]:
# ===== 年淨值走勢圖 =====
print("\n【第 6.2.5 步】繪製年淨值走勢圖...")

# 計算每年度的淨值
nav_annual = nav.resample('Y').last()
nav_annual.index = nav_annual.index.year

# 初始值設置
nav_annual_values = [1.0]  # 2000年初始淨值
for i in range(len(nav_annual) - 1):
    nav_annual_values.append(nav_annual.iloc[i])

nav_annual_values = nav_annual.values
years = nav_annual.index.astype(str).astype(int).tolist()

# 繪製年淨值走勢折線圖
fig_nav_annual = go.Figure()

# 添加淨值線
fig_nav_annual.add_trace(go.Scatter(
    x=years,
    y=nav_annual_values,
    mode='lines+markers',
    name='年末淨值',
    line=dict(color='#1f77b4', width=3),
    marker=dict(size=8, symbol='circle'),
    fill='tozeroy',
    fillcolor='rgba(31, 119, 180, 0.2)',
    hovertemplate='%{x}年<br>淨值: %{y:.4f}<extra></extra>'
))

# 添加年初淨值 (初始值為1)
years_with_initial = [years[0] - 1] + years
nav_values_with_initial = [1.0] + nav_annual_values.tolist()

fig_nav_annual.add_trace(go.Scatter(
    x=years_with_initial,
    y=nav_values_with_initial,
    mode='lines+markers',
    name='累積淨值',
    line=dict(color='#ff7f0e', width=2, dash='dash'),
    marker=dict(size=6),
    hovertemplate='%{x}年<br>累積淨值: %{y:.4f}<extra></extra>',
    showlegend=False
))

# 更新佈局
fig_nav_annual.update_layout(
    title='SSD 配對交易系統 - 年淨值走勢圖 (2000-2025)',
    xaxis_title='年份',
    yaxis_title='淨值',
    height=500,
    template='plotly_dark',
    hovermode='x unified',
    xaxis=dict(
        tickmode='linear',
        tick0=2000,
        dtick=1
    ),
    yaxis=dict(
        gridcolor='rgba(128, 128, 128, 0.2)'
    ),
    font=dict(size=11),
    showlegend=True,
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.99,
        xanchor='left',
        x=0.01,
        bgcolor='rgba(0, 0, 0, 0.5)'
    )
)

# 添加數值標籤
for year, nav_val in zip(years, nav_annual_values):
    fig_nav_annual.add_annotation(
        x=year,
        y=nav_val,
        text=f'{nav_val:.2f}',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1,
        arrowcolor='#1f77b4',
        ax=0,
        ay=-20,
        font=dict(size=9, color='#1f77b4')
    )

fig_nav_annual.show()

print("✓ 年淨值走勢圖已繪製")

# 計算年度變化統計
print("\n【年度淨值統計】")
print(f"{'年份':<8} {'年末淨值':<12} {'年度報酬(%)':<12} {'累計報酬(%)':<12}")
print("-" * 46)

prev_nav = 1.0
for year, nav_val in zip(years, nav_annual_values):
    annual_return = (nav_val - prev_nav) / prev_nav * 100
    cumulative_return = (nav_val - 1) * 100
    print(f"{year:<8} {nav_val:<12.4f} {annual_return:>10.2f}% {cumulative_return:>10.2f}%")
    prev_nav = nav_val

# 最佳年份和最差年份
best_year_idx = np.argmax(nav_annual_values)
worst_year_idx = np.argmin(nav_annual_values)

print(f"\n【極值統計】")
print(f"  最佳年份: {years[best_year_idx]} (淨值: {nav_annual_values[best_year_idx]:.4f})")
print(f"  最差年份: {years[worst_year_idx]} (淨值: {nav_annual_values[worst_year_idx]:.4f})")
print(f"  年均淨值: {np.mean(nav_annual_values):.4f}")
print(f"  年末淨值: {nav_annual_values[-1]:.4f} (總增長: {(nav_annual_values[-1]-1)*100:.2f}%)")


【第 6.2.5 步】繪製年淨值走勢圖...


✓ 年淨值走勢圖已繪製

【年度淨值統計】
年份       年末淨值         年度報酬(%)      累計報酬(%)     
----------------------------------------------
2019     1.0000             0.00%       0.00%
2020     1.0274             2.74%       2.74%
2021     0.9967            -2.99%      -0.33%
2022     1.0216             2.51%       2.16%

【極值統計】
  最佳年份: 2020 (淨值: 1.0274)
  最差年份: 2021 (淨值: 0.9967)
  年均淨值: 1.0114
  年末淨值: 1.0216 (總增長: 2.16%)


### 【窗口淨值走勢圖】
按交易窗口展示淨值變化，清晰呈現策略在不同滾動窗口期間的表現。

In [11]:
# ===== 窗口淨值走勢圖 =====
print("\n【第 6.2.6 步】繪製窗口淨值走勢圖...")

# 計算每個窗口的累積淨值
window_navs = {}
num_tranches = CAPITAL_TRANCHES
tranche_capital = INITIAL_CAPITAL / num_tranches

for col in pnl_df.columns:
    window_pnl = pnl_df[col]
    # 計算該窗口的累積報酬
    window_returns = window_pnl / tranche_capital
    window_nav = (1 + window_returns).cumprod()
    window_navs[col] = window_nav

# 繪製所有窗口的淨值曲線
fig_window_nav = go.Figure()

# 取顏色顏色列表
import plotly.express as px
colors = px.colors.qualitative.Plotly

for idx, (col, nav_series) in enumerate(window_navs.items()):
    color = colors[idx % len(colors)]
    
    fig_window_nav.add_trace(go.Scatter(
        x=nav_series.index,
        y=nav_series.values,
        mode='lines',
        name=col,
        line=dict(width=1.5),
        opacity=0.7,
        hovertemplate=f'{col}<br>日期: %{{x|%Y-%m-%d}}<br>淨值: %{{y:.4f}}<extra></extra>'
    ))

fig_window_nav.update_layout(
    title='SSD 配對交易系統 - 各窗口淨值走勢圖',
    xaxis_title='日期',
    yaxis_title='窗口淨值',
    height=600,
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(
        orientation='v',
        yanchor='top',
        y=0.99,
        xanchor='right',
        x=0.99,
        bgcolor='rgba(0, 0, 0, 0.5)',
        font=dict(size=8)
    )
)

fig_window_nav.show()

print("✓ 窗口淨值走勢圖已繪製")

# 窗口績效統計表
print("\n【窗口淨值統計】")
window_stats = []

for col, nav_series in window_navs.items():
    # 去除全為 1 的初始值
    active_nav = nav_series[nav_series != 1.0]
    
    if len(active_nav) > 0:
        final_nav = active_nav.iloc[-1]
        max_nav = active_nav.max()
        min_nav = active_nav.min()
        window_return = (final_nav - 1) * 100
        
        window_stats.append({
            '窗口': col,
            '終值': round(final_nav, 4),
            '最高': round(max_nav, 4),
            '最低': round(min_nav, 4),
            '報酬(%)': round(window_return, 2)
        })

window_stats_df = pd.DataFrame(window_stats)
if len(window_stats_df) > 0:
    print(f"共 {len(window_stats_df)} 個窗口")
    print(f"  平均淨值: {window_stats_df['終值'].mean():.4f}")
    print(f"  最佳窗口: {window_stats_df.loc[window_stats_df['報酬(%)'].idxmax(), '窗口']} "
          f"({window_stats_df['報酬(%)'].max():.2f}%)")
    print(f"  最差窗口: {window_stats_df.loc[window_stats_df['報酬(%)'].idxmin(), '窗口']} "
          f"({window_stats_df['報酬(%)'].min():.2f}%)")
    
    # 展示前 10 個窗口
    print("\n【前 10 個窗口績效】")
    display(window_stats_df.head(10).style.format({'終值': '{:.4f}', '最高': '{:.4f}', 
                                                    '最低': '{:.4f}', '報酬(%)': '{:.2f}'}))

print("✓ 窗口績效分析完成")


【第 6.2.6 步】繪製窗口淨值走勢圖...


✓ 窗口淨值走勢圖已繪製

【窗口淨值統計】
共 32 個窗口
  平均淨值: 1.0052
  最佳窗口: W001 (8.11%)
  最差窗口: W011 (-2.86%)

【前 10 個窗口績效】


,窗口,終值,最高,最低,報酬(%)
0,W000,1.0264,1.0548,0.9660,2.64
1,W001,1.0811,1.0875,0.9765,8.11
2,W002,1.0725,1.0829,0.9899,7.25
3,W003,1.0002,1.0366,0.9807,0.02
4,W004,1.0039,1.0313,0.9699,0.39
5,W005,1.0046,1.0287,0.9847,0.46
6,W006,0.9967,1.0112,0.9653,-0.33
7,W007,0.9951,1.0068,0.9564,-0.49
8,W008,0.9786,1.0221,0.9776,-2.14
9,W009,1.0264,1.0359,0.9991,2.64


✓ 窗口績效分析完成


## 進階分析：時間與部門分解

## 數據匯出與存檔

In [12]:
# === 匯出回測結果 ===
print("\n【數據匯出】")

# 確保 results 目錄存在
os.makedirs('results', exist_ok=True)

# 1. 導出 PnL
pnl_export = pnl_df.sum(axis=1)
pnl_export.to_csv('results/ssd_pnl.csv')
print("✓ 已匯出日度損益 -> results/ssd_pnl.csv")

# 2. 導出淨值
nav_export = (1 + (pnl_export / INITIAL_CAPITAL)).cumprod()
nav_export.to_csv('results/ssd_nav.csv')
print("✓ 已匯出淨值曲線 -> results/ssd_nav.csv")

# 3. 導出交易日誌
if trades:
    trades_df_export = pd.DataFrame(trades)
    trades_df_export.to_csv('results/ssd_trades.csv', index=False)
    print(f"✓ 已匯出交易日誌 ({len(trades_df_export)} 筆) -> results/ssd_trades.csv")

# 4. 導出視窗績效
windows_df_export = pd.DataFrame(backtest_result['windows'])
windows_df_export.to_csv('results/ssd_windows.csv', index=False)
print(f"✓ 已匯出視窗績效 ({len(windows_df_export)} 個) -> results/ssd_windows.csv")

# 5. 導出績效指標
if metrics:
    pd.DataFrame([metrics]).to_csv('results/ssd_metrics.csv', index=False)
    print("✓ 已匯出績效指標 -> results/ssd_metrics.csv")

print("\n✅ 回測完整！所有結果已保存到 results/ 目錄")


【數據匯出】
✓ 已匯出日度損益 -> results/ssd_pnl.csv
✓ 已匯出淨值曲線 -> results/ssd_nav.csv
✓ 已匯出交易日誌 (484 筆) -> results/ssd_trades.csv
✓ 已匯出視窗績效 (32 個) -> results/ssd_windows.csv
✓ 已匯出績效指標 -> results/ssd_metrics.csv

✅ 回測完整！所有結果已保存到 results/ 目錄
